In [ ]:
# Basis
from pathlib import Path
import numpy as np
import geopandas as gpd
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import contextily as cx
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# and from hydrolib-core
from hydrolib.core.dimr.models import DIMR, FMComponent
from hydrolib.core.dflowfm.inifield.models import IniFieldModel, DiskOnlyFileModel
from hydrolib.core.dflowfm.onedfield.models import OneDFieldModel
from hydrolib.core.dflowfm.structure.models import StructureModel
from hydrolib.core.dflowfm.crosssection.models import CrossLocModel, CrossDefModel
from hydrolib.core.dflowfm.ext.models import ExtModel
from hydrolib.core.dflowfm.mdu.models import FMModel
from hydrolib.core.dflowfm.friction.models import FrictionModel
from hydrolib.core.dflowfm.obs.models import ObservationPointModel
from hydrolib.core.dflowfm.storagenode.models import StorageNodeModel

In [ ]:
from hydrolib.dhydamo.core.hydamo import HyDAMO
from hydrolib.dhydamo.converters.df2hydrolibmodel import Df2HydrolibModel
from hydrolib.dhydamo.geometry import mesh
from hydrolib.dhydamo.core.drr import DRRModel
from hydrolib.dhydamo.core.drtc import DRTCModel
from hydrolib.dhydamo.io.dimrwriter import DIMRWriter
from hydrolib.dhydamo.io.drrwriter import DRRWriter
from hydrolib.dhydamo.geometry.viz import plot_network
from meshkernel.py_structures import DeleteMeshOption

Define in- and output paths

In [ ]:
# path to the package containing the dummy-data
data_path = Path("..\\..\\WRIJ_RR_Unpaved_methode_01_data\\data_test_rr\\")
assert data_path.exists()

In [ ]:
# overwrite output-path to write the models
output_path = Path("..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\model_alleen_rr\\")
if not output_path.exists():
    output_path.mkdir(parents=True)

## Read HyDAMO DAMO2.2 data

Explore the geopackage

In [ ]:
# all data is contained in one geopackage called 'Example model'
gpkg_file = str(data_path / "Example_model.gpkg")

# initialize a hydamo object
hydamo = HyDAMO(extent_file=data_path / "Oostrumschebeek_extent.shp")

# show content
hydamo.branches.show_gpkg(gpkg_file)

Load branches and profiles.

In the funtions below, the function 'snap_to_branch_and_drop' compares each object with a geometry to the branches. If the object is outside the specified maximum distance to any branch, the object and all objects related to it are dropped (if 'drop_related' is True).

Moreover there are multiple options to snap:
- overal: for points, based on minimum distance to the branch;
- centroid: for lines and polygons, based on the mininimum distance of the objets' centroid to the branch;
- intersecting: for lines, takes the first branch the object is intersecting (for lines);
- ends: for lines, based on the cumulative distance of the lines' ends to the branch.

In [ ]:
hydamo.branches.read_gpkg_layer(gpkg_file, layer_name="HydroObject", index_col="code")

Catchments and laterals

In [ ]:
# read catchments
hydamo.catchments.read_gpkg_layer(gpkg_file, layer_name="afvoergebiedaanvoergebied", index_col="code", check_geotype=False)

In [ ]:
hydamo.catchments

In [ ]:
# read laterals
hydamo.laterals.read_gpkg_layer(gpkg_file, layer_name="lateraleknoop")
hydamo.laterals.snap_to_branch(hydamo.branches, snap_method="overal", maxdist=5000)
hydamo.catchments['boundary_node'] = [hydamo.laterals[hydamo.laterals.globalid==c['lateraleknoopid']].code.values[0] for _,c in hydamo.catchments.iterrows()]

## Rainfall runoff model

RR has not changed yet compared to delft3dfmpy. Initialize a model:

In [ ]:
drrmodel = DRRModel()

Catchments are provided in the HyDAMO DAMO2.2 format and included in the GPKG. They can also be read from other formats using 'read_gml', or 'read_shp'. Note that in case of shapefiles column mapping is necessary because the column names are truncated. 

Note that when catchments have a "MultiPolygon' geometry, the multipolygons are 'exploded' into single polygon geometries. A warning of this is isued, and a suffix is added to every polygons ID to prevent duplicates. 

For every catchment, the land use areas will be calculated and if appopriate a maximum of four RR-nodes will be created per catchment:
 - unpaved (based on the Ernst concept)
 - paved 
 - greenhouse
 - open water (not the full Sobek2 open water, but only used to transfer (net) precipitation that falls on open water that is schematized in RR to the 1D/2D network.
 
At the moment, two options exist for the schematisation of the paved area:
 1) simple: the paved fraction of each catchment is modelled with a paved node, directly connected to catchments' boundary node
 <br>
 2) more complex: sewer area polygons and overflow points are used a input as well. For each sewer area, the overlapping paved area is the distributed over the overflows that are associated with the sewerarea (the column 'lateraleknoopcode') using the area fraction (column 'fractie') for each overflow. In each catchment, paved area that does not intersect with a sewer area gets an unpaved node as in option (1). <br>

Similar to sewer areas, additional greenhouse polygons and the corresponding laterals can be added to the model. For example, when these areas and their inflow locations are exactly known. 

Load data and settings. RR-parameters can be derived from a raster (using zonal statistics per catchment), or provided as a standard number. Rasters can be in any format that is accepted by the package rasterio: https://gdal.org/drivers/raster/index.html. All common GIS-formats (.asc, .tif) are accepted.

In [ ]:
lu_file = data_path / "rasters" / "sobek_landuse.tif"
ahn_file = data_path / "rasters" / "AHN_2m_clipped_filled.tif"
soil_file = data_path / "rasters" / "sobek_soil.tif"
surface_storage = 10.0 # [mm]
infiltration_capacity = 100.0 # [mm/hr]
initial_gwd = 1.2  # water level depth below surface [m]  
runoff_resistance = 5.0 # [d]
infil_resistance = 300.0 # [d]
layer_depths = [0.0, 1.0, 2.0] # [m]
layer_resistances = [300, 2000, 100000] # [d]

A different meteo-station can be assigned to each catchment, of a different shape can be provided. Here, 'meteo_areas' are assumed equal to the catchments.

In [ ]:
meteo_areas = hydamo.catchments 

### Unpaved nodes

For land use and soil type a coding is prescribed. For landuse, the legend of the map is expected to be as follows: <br>
 1 potatoes  <br>
 2 wheat<br>
 3 sugar beet<br> 
 4 corn       <br> 
 5 other crops <br> 
 6 bulbous plants<br> 
 7 orchard<br>
 8 grass  <br>
 9 deciduous forest  <br>
10 coniferous forest<br>
11 nature<br>
12 barren<br>
13 open water<br>
14 built-up<br>
15 greenhouses<br>

For classes 1-12, the areas are calculated from the provided raster and remapped to the classification in the Sobek RR-tables.


The coding for the soil types:<br>
1 'Veengrond met veraarde bovengrond'<br>
 2 'Veengrond met veraarde bovengrond, zand'<br>
 3 'Veengrond met kleidek'<br>
 4 'Veengrond met kleidek op zand'<br>
 5 'Veengrond met zanddek op zand'<br>
 6 'Veengrond op ongerijpte klei'<br>
 7 'Stuifzand'<br>
 8 'Podzol (Leemarm, fijn zand)'<br>
 9 'Podzol (zwak lemig, fijn zand)'<br>
10 'Podzol (zwak lemig, fijn zand op grof zand)'<br>
11 'Podzol (lemig keileem)'<br>
12 'Enkeerd (zwak lemig, fijn zand)'<br>
13 'Beekeerd (lemig fijn zand)'<br>
14 'Podzol (grof zand)'<br>
15 'Zavel'<br>
16 'Lichte klei'<br>
17 'Zware klei'<br>
18 'Klei op veen'<br>
19 'Klei op zand'<br>
20 'Klei op grof zand'<br>
21 'Leem'<br>


And surface elevation needs to be in m+NAP.

Unpaved_from_input has now an optional argument containing the greenhouse areas: if a catchment intersects them its area (the most ocurring class) is corrected for the greenhouse area. 

In [ ]:
drrmodel.unpaved.io.unpaved_from_input(
    hydamo.catchments,
    lu_file,
    ahn_file,
    soil_file,
    surface_storage,
    infiltration_capacity,
    initial_gwd,
    meteo_areas,
    greenhouse_areas=hydamo.greenhouse_areas,
)
drrmodel.unpaved.io.ernst_from_input(
    hydamo.catchments,
    depths=layer_depths,
    resistance=layer_resistances,
    infiltration_resistance=infil_resistance,
    runoff_resistance=runoff_resistance,
)

### Open water

As opposed to Sobek, in D-Hydro open water is merely an interface for precpitation and evaporation. No management and water levels are included.

In [ ]:
# RR
drrmodel.openwater.io.openwater_from_input(
    hydamo.catchments, lu_file, meteo_areas, zonalstats_alltouched=True
)

### RR boundaries

They are different for the (paved) case with and without overflows. Overflows and greenhouse laterals are optional, but should be provided if they have been used above. 

In [ ]:
drrmodel.external_forcings.io.boundary_from_input(
    hydamo.laterals, 
    hydamo.catchments, 
    drrmodel, 
    overflows=hydamo.overflows, 
    #greenhouse_laterals=hydamo.greenhouse_laterals
)

### External forcings

Three types of external forcing need to be provided:<br>
- Seepage/drainage
- Precipitation
- Evaporation

All are assumed to be spatially variable and thus need to pe provided as rasters per time step. Only the locations of the folders containing the rasters need to be provided; the time step is then derived from the file names.

Precipitation and evaporation are assumed to be in mm/d. As for evaporation only one meteostation is used, the meteo_areas are dissolved. For seepage, as the use of Metaswap-rasters is allowed, the unit is assumed to m3/grid cell/timestep.

Rastertypes can be any type that is recognized by rasterio (in any case Geotiff and ArcASCII rasters). If the file extension is 'IDF', as is the case in Modflow output, the raster is read using the 'imod'-package.

IMPORTANT: time steps are extracted from the file names. Therefore, the names should cohere to some conditions:
The filename should consist of at least two parts, separated by underscores. The second part needs to contain time information, which should be formatted as YYYYMMDDHHMMSS (SS may be omitted). Or, for daily data YYYYMMDD.

For example: 'precip_20200605151500.tif'

Extracting meteo-data from rasters can be time consuming. If precip_file and evap_file are specified, meteo-files are copied from an existing location.

In [ ]:
seepage_folder = data_path / "rasters" / "seepage"
precip_file = str(data_path / "DEFAULT.BUI")
evap_folder = data_path / "rasters" / "evaporation"
drrmodel.external_forcings.io.seepage_from_input(hydamo.catchments, seepage_folder)
drrmodel.external_forcings.io.precip_from_input(meteo_areas, precip_folder=None, precip_file=precip_file)
drrmodel.external_forcings.io.evap_from_input(meteo_areas, evap_folder=evap_folder, evap_file=None)

Add the main parameters:

In [ ]:
drrmodel.d3b_parameters["Timestepsize"] = 300
drrmodel.d3b_parameters["StartTime"] = "'2016/06/01;00:00:00'"  # should be equal to refdate for D-HYDRO
drrmodel.d3b_parameters["EndTime"] = "'2016/06/03;00:00:00'"
drrmodel.d3b_parameters["RestartIn"] = 0
drrmodel.d3b_parameters["RestartOut"] = 0
drrmodel.d3b_parameters["RestartFileNamePrefix"] = "Test"
drrmodel.d3b_parameters["UnsaturatedZone"] = 1
drrmodel.d3b_parameters["UnpavedPercolationLikeSobek213"] = -1
drrmodel.d3b_parameters["VolumeCheckFactorToCF"] = 100000

Laterals are different for the case with and without RR. There can be three options:
1) laterals from the RR model (RR=True). There will be real-time coupling where RR and FM are calculated in parallel. Note that, again, the overflows are needed because there are extra boundaries. If there are no overflows, it does not have to be provided.
2) timeseries: lateral_discharges can be a dataframe with the code of the lateral as column headers and timesteps as index
3) constant: lateral_discharges can be a pandas Series with the code of the lateral as the index. This is the case in the example when RR=False.

In [ ]:
hydamo.external_forcings.convert.laterals(
    hydamo.laterals,
    overflows=hydamo.overflows,
    #greenhouse_laterals=hydamo.greenhouse_laterals,
    lateral_discharges=None,
    rr_boundaries=drrmodel.external_forcings.boundary_nodes
)

### Plot the RR model

In [ ]:
def node_geometry(dict):
    # Function to put the node geometries in geodataframes
    from shapely.geometry import Point, LineString

    geoms = []
    links = []
    for i in dict.items():
        if "ar" in i[1]:
            if np.sum([float(s) for s in i[1]["ar"].split(" ")]) > 0:
                geoms.append(Point((float(i[1]["px"]), float(i[1]["py"]))))
                links.append(
                    LineString(
                        (
                            Point(float(i[1]["px"]), float(i[1]["py"])),
                            Point(
                                float(drrmodel.external_forcings.boundary_nodes[i[1]["boundary_node"]]["px"]),
                                float(drrmodel.external_forcings.boundary_nodes[i[1]["boundary_node"]]["py"]),
                            ),
                        )
                    )
                )
        else:
            geoms.append(Point((float(i[1]["px"]), float(i[1]["py"]))))
    return ((gpd.GeoDataFrame(geoms, columns=["geometry"])), gpd.GeoDataFrame(links, columns=["geometry"]))

In [ ]:
plt.rcParams['axes.edgecolor'] = 'w'
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 8))

ax.xaxis.set_visible(False)
ax.yaxis.set_visible(False)
xmin,ymin,xmax,ymax=hydamo.clipgeo.bounds
ax.set_xlim(round(xmin), round(xmax))
ax.set_ylim(round(ymin), round(ymax))

hydamo.catchments.geometry.plot(ax=ax, label="Catchments", edgecolor="black", facecolor="pink", alpha=0.5)
hydamo.branches.geometry.plot(ax=ax, label="Channel")
node_geometry(drrmodel.unpaved.unp_nodes)[0].plot(
    ax=ax, markersize=30, marker="s", color="green", label="Unpaved"
)
node_geometry(drrmodel.unpaved.unp_nodes)[1].plot(ax=ax, color="black", linewidth=0.5)
node_geometry(drrmodel.paved.pav_nodes)[0].plot(ax=ax, markersize=20, marker="s", color="red", label="Paved")
node_geometry(drrmodel.paved.pav_nodes)[1].plot(ax=ax, color="black", linewidth=0.5)
node_geometry(drrmodel.greenhouse.gh_nodes)[0].plot(ax=ax, markersize=15, color="yellow", label="Greenhouse")
node_geometry(drrmodel.greenhouse.gh_nodes)[1].plot(ax=ax, color="black", linewidth=0.5)
node_geometry(drrmodel.openwater.ow_nodes)[0].plot(ax=ax, markersize=10, color="blue", label="Openwater")
node_geometry(drrmodel.openwater.ow_nodes)[1].plot(ax=ax, color="black", linewidth=0.5, label="RR-link")
node_geometry(drrmodel.external_forcings.boundary_nodes)[0].plot(
    ax=ax, markersize=15, color="purple", label="RR Boundary"
)

# manually add handles for polygon plot
handles, labels = ax.get_legend_handles_labels()
poly = mpatches.Patch(facecolor="pink", edgecolor="black", alpha=0.5)    
ax.legend(handles=handles.append(poly), labels=labels.append("Catchments"))
cx.add_basemap(ax, crs=28992, source=cx.providers.OpenStreetMap.Mapnik)
fig.tight_layout()

## Writing the model

In D-Hydro the 1D timestep (dt user) should be at least equal to than the smallest timestep of RR and RTC, otherwise water balance problems may occur.
The following code sets 'dtuser' equal to the smallest time step.

In [ ]:
# check the timesteps:
timesteps = []
timesteps.append(drrmodel.d3b_parameters['Timestepsize'])

In [ ]:
rr_writer = DRRWriter(drrmodel, output_dir=output_path, name="test", wwtp=(199000.0, 396000.0))
rr_writer.write_all()

A run.bat that will run DIMR is written by the following command. Adjust this with your local D-Hydro Suite version.

In [ ]:
# dimr = DIMRWriter(output_path=output_path, dimr_path=str(r"C:\Program Files\Deltares\D-HYDRO Suite 2024.03 1D2D\plugins\DeltaShell.Dimr\kernels\x64\bin\run_dimr.bat"))

In [ ]:
# if not RR:
#     drrmodel = None
# if not RTC:
#     drtcmodel = None

In [ ]:
# dimr.write_dimrconfig(fm, rr_model=drrmodel, rtc_model=drtcmodel)

Add projection information (Rijksdriehoeksstelsel) to the net.nc-file.

In [ ]:
# dimr.add_crs()

In [ ]:
# dimr.write_runbat()

In [ ]:
# print("Done!")